# Homework 3  
Sarayu Rao

##1. Building a model

Main Question: What are the best predictors of a driver's finishing position for a given race? 

###Data Preparation
<br>
Combining pitstops and results dataset to create my final dataset for evaluation.

In [0]:
#Pyspark Imports
from pyspark.sql.functions import col, round, avg, upper, substring, when, length, floor, datediff, current_date, max, min, sum, when, regexp_extract
import pyspark.sql.functions as F
import pandas as pd

#ML imports
from sklearn.model_selection import train_test_split 
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
import os
import matplotlib.pyplot as plt
import mlflow.sklearn
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error, mean_absolute_percentage_error, explained_variance_score, median_absolute_error
import tempfile


In [0]:
#Load pitstop dataset
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

#Case necessary columns to integers
df_pitstops = df_pitstops.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "milliseconds": col("milliseconds").cast("int")})

#Get each driver's average pitstop time for each race
df_avg_pit = df_pitstops.groupBy("raceId","driverId").avg("milliseconds")


In [0]:
#Load results dataset
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

#Join average pitstop time for each driver's race and rename column
df_race_results = df_results.join(df_avg_pit, on = ["raceId","driverId"])
df_race_results = df_race_results.withColumnRenamed("avg(milliseconds)", "avgPitstop")
display(df_race_results)


In [0]:
#Keep and cast necessary columns to integers for our model
df_race_results = df_race_results.select(["raceId","driverId","resultId","positionOrder","laps","fastestLap","fastestLapTime","fastestLapSpeed","avgPitstop","grid", "rank","fastestLapTime"])

df_race_results = df_race_results.filter((df_race_results["fastestLap"] != "\\N") & (df_race_results["fastestLapTime"] != "\\N") & (df_race_results["fastestLapSpeed"] != "\\N"))
df_race_results = df_race_results.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "resultId": col("resultId").cast("int"),
                                      "positionOrder": col("positionOrder").cast("int"),
                                      "laps": col("laps").cast("int"),
                                      "fastestLap": col("fastestLap").cast("int"),
                                      "fastestLapSpeed": col("fastestLapSpeed").cast("float"),
                                      "avgPitstop": col("avgPitstop").cast("float"),
                                      "grid": col("grid").cast("int"),
                                      "rank":col("rank").cast("int"),
                                     })

#Converting fastest lap time to miliseconds 
df_race_results = df_race_results.withColumn("fastestLapTime_ms",
    when(col("fastestLapTime") != "\\N",
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 1).cast("int") * 60000) +  # minutes
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 2).cast("int") * 1000) +  # seconds
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 3).cast("int") * 1)      # milliseconds
    ).otherwise(None)
)
display(df_race_results)

In [0]:
#Converting dataset to a pandas dataset and assigning my X and y variables
df_pandas = df_race_results.toPandas()
X = df_pandas[['laps', 'fastestLap', 'fastestLapSpeed', 'avgPitstop', 'grid', 'rank', 'fastestLapTime_ms']]
y = df_pandas[['positionOrder']]

#Train, test, split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

#Format y values properly
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()


###Setting up experiment  

1. Setting up initial experiment to get ExperimentID  
2. Defining mlflow function to perform multiple experiments with ease
3. Run an initial experiment to test that outputs are as expected

In [0]:
#-----------------Initial Experiment-------------------------------
#Running initial flow to get experiementID to use in future log function

with mlflow.start_run(run_name="Initial RF Experiment") as run:
  # Create model, train it, and create predictions
  rf = RandomForestRegressor()
  rf.fit(X_train, y_train)
  #Make sure positionOrder is a whole number between 1 and 30
  predictions = rf.predict(X_test).round().astype(int).clip(1, 30)
  
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-regressor")
  
  # Create metrics
  mse = mean_squared_error(y_test, predictions)
  mae = mean_absolute_error(y_test, predictions)
  r2 = r2_score(y_test, predictions)
  rmse = root_mean_squared_error(y_test, predictions)
  mape = mean_absolute_percentage_error(y_test, predictions)
  explained_variance = explained_variance_score(y_test, predictions)
  mdae = median_absolute_error(y_test, predictions)
  
  print("  mse: {}".format(mse))
  print("  mae: {}".format(mae))
  print("  r2: {}".format(r2))
  print("  rmse: {}".format(rmse))
  print("  mape: {}".format(mape))
  print("  explained_variance: {}".format(explained_variance))
  print("  mdae: {}".format(mdae))
  
  # Log metrics
  mlflow.log_metric("mse", mse)
  mlflow.log_metric("mae", mae)
  mlflow.log_metric("r2", r2)
  mlflow.log_metric("rmse", rmse)
  mlflow.log_metric("mape", mape)
  mlflow.log_metric("explained_variance", explained_variance)
  mlflow.log_metric("mdae", mdae)

  #Get runID and experimentID
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:
#-----------------Define Function-------------------------------
#Defining function to run multiple experiments with ease 
def f1_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test):

  with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
    # Create model, train it, and create predictions
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    #Make sure positionOrder is a whole number between 1 and 20
    predictions = rf.predict(X_test).round().astype(int).clip(1, 30)

    # Log model
    mlflow.sklearn.log_model(rf, "random-forest-regressor")

    # Log params
    [mlflow.log_param(param, value) for param, value in params.items()]

    # Create metrics
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    rmse = root_mean_squared_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    explained_variance = explained_variance_score(y_test, predictions)
    mdae = median_absolute_error(y_test, predictions)
    
    print("  mse: {}".format(mse))
    print("  mae: {}".format(mae))
    print("  r2: {}".format(r2))
    print("  rmse: {}".format(rmse))
    print("  mape: {}".format(mape))
    print("  explained_variance: {}".format(explained_variance))
    print("  mdae: {}".format(mdae))
    
    # Log metrics
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mape", mape)
    mlflow.log_metric("explained_variance", explained_variance)
    mlflow.log_metric("mdae", mdae)

    
    # Create feature importance
    importance = pd.DataFrame(list(zip(X.columns, rf.feature_importances_)), 
                                columns=["Feature", "Importance"]
                              ).sort_values("Importance", ascending=False)
    
    # Log importances using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
    temp_name = temp.name
    try:
      importance.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")
    finally:
      temp.close() # Delete the temp file
    
    #Create plot showing actual vs predicted values
    
    fig, ax = plt.subplots()
    ax.scatter(y_test, predictions, alpha=0.5)
    #Perfect prediction line
    ax.plot([1, 20], [1, 20], 'r--')  

    ax.set_xlabel("Actual Position")
    ax.set_ylabel("Predicted Position")
    ax.set_title("Actual vs Predicted Position")


    #Log plot using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="actual_v_predict", suffix=".png")
    temp_name = temp.name
    try:
      fig.savefig(temp_name)
      mlflow.log_artifact(temp_name, "actual_v_predict.png")
    finally:
      temp.close() # Delete the temp file
      
    display(fig)
    return run.info.run_id

In [0]:
#-----------------Testing Outputs-------------------------------
#Testing outputs 
params = {
  "n_estimators": 100,
  "max_depth": 5,
  "random_state": 42
}

f1_rf(experimentID, "5th Testing Flow Outputs", params, X_train, X_test, y_train, y_test)

###Running Experiments
<br>
Running a total of 14 experiments. The name of each experiment differentiates the parameters for each run

In [0]:
#1
params = {
  "n_estimators": 100,
  "max_depth": 5,
  "random_state": 42
}

f1_rf(experimentID, "100_estimators_5_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#2
params = {
  "n_estimators": 100,
  "max_depth": 10,
  "random_state": 42
}
f1_rf(experimentID, "100_estimators_10_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#3
params = {
  "n_estimators": 100,
  "max_depth": 20,
  "random_state": 42
}
f1_rf(experimentID, "100_estimators_20_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#4
params = {
  "n_estimators": 200,
  "max_depth": 5,
  "random_state": 42
}
f1_rf(experimentID, "200_estimators_5_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#5
params = {
  "n_estimators": 200,
  "max_depth": 10,
  "random_state": 42
}
f1_rf(experimentID, "200_estimators_10_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#6
params = {
  "n_estimators": 200,
  "max_depth": 20,
  "random_state": 42
}
f1_rf(experimentID, "200_estimators_20_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#7
params = {
  "n_estimators": 500,
  "max_depth": 20,
  "random_state": 42
}
f1_rf(experimentID, "500_estimators_20_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#8
params = {
  "n_estimators": 1000,
  "max_depth": 20,
  "random_state": 42
}
f1_rf(experimentID, "1000_estimators_20_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#9
params = {
  "n_estimators": 500,
  "max_depth": 30,
  "random_state": 42
}
f1_rf(experimentID, "500_estimators_30_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#10
params = {
  "n_estimators": 350,
  "max_depth": 25,
  "random_state": 42
}
f1_rf(experimentID, "350_estimators_25_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#11
params = {
  "n_estimators": 200,
  "max_depth": 25,
  "random_state": 42
}
f1_rf(experimentID, "200_estimators_25_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#12
params = {
  "n_estimators": 300,
  "max_depth": 35,
  "random_state": 42
}
f1_rf(experimentID, "300_estimators_35_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#13
params = {
  "n_estimators": 1500,
  "max_depth": 50,
  "random_state": 42
}
f1_rf(experimentID, "1500_estimators_50_depth", params, X_train, X_test, y_train, y_test)

In [0]:
#14
params = {
  "n_estimators": 2000,
  "max_depth": 100,
  "random_state": 42
}
f1_rf(experimentID, "2000_estimators_100_depth", params, X_train, X_test, y_train, y_test)

##Best Models  
<br>
Looking at all 14 runs, the model that performed the best had 2000 estimators and a depth of 100. The main metric for model comparison in random forest regression is the MSE (mean squared error), and this model had the lowest MSE of 9.79 and root MSE of 3.129. The RMSE, MAE, and MAPE tell us that our model is typically off by about 2-3 positions, and our r-squared and explained variance tells us that about 72% of variance in positionOrder is explained by our model.
<br>
<br>
However, I would also point out that our run that used 350 estimators and a depth of 25 produced results comparable to our best performing model, but uses significantly less estimators. This means that we would be able to get results close enough to the best model with lower run costs, which is important to take into consideration for larger-scale production. 